# 从零实现 GIN 图分类：可学习 $\epsilon$、多图 Batch 与图级读出

本 Notebook 只使用 PyTorch 基础张量与 `nn.Module`，手写 `GINLayer`、`GINClassifier`、无 PyG 的多图拼接以及图级 sum pooling；不使用 `torch_geometric`、DGL 或任何现成 GNN 层。

学习目标不是背一段网络定义，而是打通一条可审计链路：图合同 → 邻居求和 oracle → `(1+ε)` 中心节点更新 → 多图 batch 隔离 → 节点置换不变性 → 受控图分类训练 → 梯度与制品合同。所有样本均为离线合成小图，固定随机种子、CPU、单线程；结果只验证实现与评估协议，不代表真实业务泛化能力。

In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import time  # 导入本单元所需的依赖。

import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 3701  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
DTYPE = torch.float32  # 计算并保存当前步骤的中间状态。

def canonical_hash(payload) -> str:  # 定义本节可复用的核心函数。
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]  # 返回当前分支计算出的结果。

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert torch.initial_seed() == SEED  # 用受控断言验证关键不变量。
assert torch.__version__ and not any(name in globals() for name in ("torch_geometric", "dgl"))  # 用受控断言验证关键不变量。

## 1. 图样本与无泄漏切分

构造两类无向图：类别 0 是环，类别 1 是星形；节点数在 6–8 之间变化，节点初始特征只含常数项和很小的、与类别无关的局部标记。分类所需的主要信号因此来自拓扑，而不是把标签偷偷写进特征。

每类 18 张图：前 12 张训练、随后 3 张验证、最后 3 张测试。`graph_id` 在三个集合中互斥；validation 只选择 checkpoint，test 在模型冻结后只打开一次。这里的受控任务用于验证 GIN 能否学习，不能被包装成真实数据基准。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class GraphSample:  # 定义承载本节状态与行为的数据结构。
    graph_id: str  # 执行当前语句以推进本节示例。
    x: torch.Tensor  # 执行当前语句以推进本节示例。
    edge_pairs: tuple[tuple[int, int], ...]  # 执行当前语句以推进本节示例。
    label: int  # 执行当前语句以推进本节示例。
    split: str  # 执行当前语句以推进本节示例。

def make_graph(label: int, index: int, split: str) -> GraphSample:  # 定义本节可复用的核心函数。
    if label not in (0, 1) or split not in {"train", "val", "test"}:  # 按当前条件选择后续控制路径。
        raise ValueError("label/split 非法")  # 遇到非法合同立即显式失败。
    n = 6 + index % 3  # 计算并保存当前步骤的中间状态。
    if label == 0:  # 按当前条件选择后续控制路径。
        pairs = {tuple(sorted((i, (i + 1) % n))) for i in range(n)}  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        pairs = {(0, i) for i in range(1, n)}  # 计算并保存当前步骤的中间状态。
    # 第二列仅是节点局部奇偶标记，不依赖图标签。
    x = torch.tensor([[1.0, 0.05 * (i % 2)] for i in range(n)], dtype=DTYPE)  # 计算并保存当前步骤的中间状态。
    return GraphSample(f"{split}-c{label}-{index:02d}", x, tuple(sorted(pairs)), label, split)  # 返回当前分支计算出的结果。

graphs = []  # 计算并保存当前步骤的中间状态。
for label in (0, 1):  # 遍历输入元素以累积或检查结果。
    for i in range(18):  # 遍历输入元素以累积或检查结果。
        split = "train" if i < 12 else ("val" if i < 15 else "test")  # 计算并保存当前步骤的中间状态。
        graphs.append(make_graph(label, i, split))  # 执行当前语句以推进本节示例。

split_graphs = {s: [g for g in graphs if g.split == s] for s in ("train", "val", "test")}  # 计算并保存当前步骤的中间状态。
all_ids = [g.graph_id for g in graphs]  # 计算并保存当前步骤的中间状态。
assert len(graphs) == 36 and len(all_ids) == len(set(all_ids))  # 用受控断言验证关键不变量。
assert tuple(len(split_graphs[s]) for s in ("train", "val", "test")) == (24, 6, 6)  # 用受控断言验证关键不变量。
assert all(set(g.label for g in split_graphs[s]) == {0, 1} for s in split_graphs)  # 用受控断言验证关键不变量。
assert not ({g.graph_id for g in split_graphs["train"]} & {g.graph_id for g in split_graphs["test"]})  # 用受控断言验证关键不变量。
assert all(g.x.shape[1] == 2 and torch.isfinite(g.x).all() for g in graphs)  # 用受控断言验证关键不变量。

## 2. 不依赖 PyG 的多图 Batch

稀疏多图 batch 不需要补齐节点：把各图节点沿第 0 维拼接，把第 $k$ 张图的边端点加上累计节点偏移，并维护 `batch[i]=k`。拼接后的邻接在数学上是块对角结构；任意一条边两端的 `batch` 必须相同，否则一张图会读取另一张图的消息。

对图级 sum pooling：$h_G=\sum_{v:\,batch(v)=G}h_v$。求和保留多重集合计数，是 GIN 表达力论证的重要组成；mean pooling 会抹去部分图规模信息。空图、越界边、重复/自环原始边、错 batch 都应 fail-closed，而不是依赖 `index_add_` 偶然报错。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class GraphBatch:  # 定义承载本节状态与行为的数据结构。
    x: torch.Tensor  # 执行当前语句以推进本节示例。
    edge_index: torch.Tensor  # 执行当前语句以推进本节示例。
    batch: torch.Tensor  # 执行当前语句以推进本节示例。
    labels: torch.Tensor  # 执行当前语句以推进本节示例。
    graph_ids: tuple[str, ...]  # 执行当前语句以推进本节示例。
    num_graphs: int  # 执行当前语句以推进本节示例。

def validate_sample(graph: GraphSample, feature_dim: int | None = None) -> None:  # 定义本节可复用的核心函数。
    if graph.x.ndim != 2 or graph.x.shape[0] == 0:  # 按当前条件选择后续控制路径。
        raise ValueError("空图或 x 不是二维张量")  # 遇到非法合同立即显式失败。
    if feature_dim is not None and graph.x.shape[1] != feature_dim:  # 按当前条件选择后续控制路径。
        raise ValueError("多图特征维度不一致")  # 遇到非法合同立即显式失败。
    if not torch.isfinite(graph.x).all():  # 按当前条件选择后续控制路径。
        raise ValueError("节点特征包含非有限值")  # 遇到非法合同立即显式失败。
    n = graph.x.shape[0]  # 计算并保存当前步骤的中间状态。
    seen = set()  # 计算并保存当前步骤的中间状态。
    for u, v in graph.edge_pairs:  # 遍历输入元素以累积或检查结果。
        if not (0 <= u < n and 0 <= v < n) or u == v:  # 按当前条件选择后续控制路径。
            raise ValueError("边端点越界或原始边含自环")  # 遇到非法合同立即显式失败。
        key = tuple(sorted((u, v)))  # 计算并保存当前步骤的中间状态。
        if key in seen:  # 按当前条件选择后续控制路径。
            raise ValueError("无向边重复")  # 遇到非法合同立即显式失败。
        seen.add(key)  # 执行当前语句以推进本节示例。

def validate_batched_graph_tensors(x: torch.Tensor, edge_index: torch.Tensor,  # 定义本节可复用的核心函数。
                                   batch: torch.Tensor, num_graphs: int) -> None:  # 执行当前语句以推进本节示例。
    if x.ndim != 2 or x.shape[0] == 0 or not torch.isfinite(x).all():  # 按当前条件选择后续控制路径。
        raise ValueError("节点特征必须是非空有限二维张量")  # 遇到非法合同立即显式失败。
    if edge_index.ndim != 2 or edge_index.shape[0] != 2 or edge_index.dtype != torch.long:  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 必须是 shape=(2,E) 的 long 张量")  # 遇到非法合同立即显式失败。
    if batch.ndim != 1 or batch.dtype != torch.long or batch.numel() != x.shape[0] or num_graphs <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("batch/num_graphs 合同不匹配")  # 遇到非法合同立即显式失败。
    if x.device != edge_index.device or x.device != batch.device:  # 按当前条件选择后续控制路径。
        raise ValueError("x/edge_index/batch 必须位于同一设备")  # 遇到非法合同立即显式失败。
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= x.shape[0]):  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 越界")  # 遇到非法合同立即显式失败。
    if int(batch.min()) < 0 or int(batch.max()) >= num_graphs:  # 按当前条件选择后续控制路径。
        raise ValueError("图编号越界")  # 遇到非法合同立即显式失败。
    counts = torch.bincount(batch, minlength=num_graphs)  # 计算并保存当前步骤的中间状态。
    if counts.numel() != num_graphs or bool((counts == 0).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("batch 声明了空图")  # 遇到非法合同立即显式失败。
    if edge_index.numel() and not torch.equal(batch[edge_index[0]], batch[edge_index[1]]):  # 按当前条件选择后续控制路径。
        raise ValueError("检测到跨图边污染")  # 遇到非法合同立即显式失败。


def batch_graphs(items: list[GraphSample]) -> GraphBatch:  # 定义本节可复用的核心函数。
    if not items:  # 按当前条件选择后续控制路径。
        raise ValueError("至少需要一张非空图")  # 遇到非法合同立即显式失败。
    feature_dim = items[0].x.shape[1]  # 计算并保存当前步骤的中间状态。
    xs, directed, owners, labels, ids = [], [], [], [], []  # 计算并保存当前步骤的中间状态。
    offset = 0  # 计算并保存当前步骤的中间状态。
    for gid, graph in enumerate(items):  # 遍历输入元素以累积或检查结果。
        validate_sample(graph, feature_dim)  # 执行当前语句以推进本节示例。
        xs.append(graph.x); labels.append(graph.label); ids.append(graph.graph_id)  # 执行当前语句以推进本节示例。
        owners.append(torch.full((graph.x.shape[0],), gid, dtype=torch.long))  # 计算并保存当前步骤的中间状态。
        for u, v in graph.edge_pairs:  # 遍历输入元素以累积或检查结果。
            directed.extend([(offset + u, offset + v), (offset + v, offset + u)])  # 执行当前语句以推进本节示例。
        offset += graph.x.shape[0]  # 计算并保存当前步骤的中间状态。
    edge_index = (torch.tensor(directed, dtype=torch.long).T.contiguous()  # 计算并保存当前步骤的中间状态。
                  if directed else torch.empty((2, 0), dtype=torch.long))  # 按当前条件选择后续控制路径。
    out = GraphBatch(torch.cat(xs), edge_index, torch.cat(owners),  # 计算并保存当前步骤的中间状态。
                     torch.tensor(labels, dtype=torch.long), tuple(ids), len(items))  # 计算并保存当前步骤的中间状态。
    validate_batched_graph_tensors(out.x, out.edge_index, out.batch, out.num_graphs)  # 执行当前语句以推进本节示例。
    return out  # 返回当前分支计算出的结果。

def sum_pool(x: torch.Tensor, batch: torch.Tensor, num_graphs: int) -> torch.Tensor:  # 定义本节可复用的核心函数。
    empty_edges = torch.empty((2, 0), dtype=torch.long, device=x.device)  # 计算并保存当前步骤的中间状态。
    validate_batched_graph_tensors(x, empty_edges, batch, num_graphs)  # 执行当前语句以推进本节示例。
    pooled = x.new_zeros((num_graphs, x.shape[1]))  # 计算并保存当前步骤的中间状态。
    pooled.index_add_(0, batch, x)  # 执行当前语句以推进本节示例。
    return pooled  # 返回当前分支计算出的结果。

probe_batch = batch_graphs([graphs[0], graphs[18]])  # 计算并保存当前步骤的中间状态。
assert probe_batch.x.shape[0] == graphs[0].x.shape[0] + graphs[18].x.shape[0]  # 用受控断言验证关键不变量。
assert torch.equal(probe_batch.batch[probe_batch.edge_index[0]], probe_batch.batch[probe_batch.edge_index[1]])  # 用受控断言验证关键不变量。
assert torch.equal(sum_pool(torch.ones(probe_batch.x.shape[0], 1), probe_batch.batch, 2).squeeze(),  # 用受控断言验证关键不变量。
                   torch.tensor([float(graphs[0].x.shape[0]), float(graphs[18].x.shape[0])]))  # 执行当前语句以推进本节示例。
try:  # 尝试执行可能失败的受控操作。
    batch_graphs([])  # 执行当前语句以推进本节示例。
    raise AssertionError("空图列表未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "至少需要" in str(exc)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    sum_pool(torch.ones(2, 3), torch.tensor([0, 0]), 2)  # 执行当前语句以推进本节示例。
    raise AssertionError("声明空图的 batch 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "空图" in str(exc)  # 用受控断言验证关键不变量。

## 3. 邻居求和：先固定消息方向

约定 `edge_index[0]=source`、`edge_index[1]=target`，因此

$$a_i=\sum_{j\in\mathcal N(i)}x_j.$$

下方 3 节点 oracle 只有无向边 `0—1—2`，输入为 `[1,2,4]`：节点 0 收到 2，节点 1 收到 $1+4=5$，节点 2 收到 2。这个断言能同时发现 source/target 颠倒、误加自环以及把 sum 写成 mean。

In [ ]:
def neighbor_sum(x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
    if x.ndim != 2 or edge_index.ndim != 2 or edge_index.shape[0] != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("x 或 edge_index shape 非法")  # 遇到非法合同立即显式失败。
    if edge_index.numel() and (int(edge_index.min()) < 0 or int(edge_index.max()) >= x.shape[0]):  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 越界")  # 遇到非法合同立即显式失败。
    source, target = edge_index  # 计算并保存当前步骤的中间状态。
    out = torch.zeros_like(x)  # 计算并保存当前步骤的中间状态。
    out.index_add_(0, target, x[source])  # 执行当前语句以推进本节示例。
    return out  # 返回当前分支计算出的结果。

oracle_x = torch.tensor([[1.0], [2.0], [4.0]])  # 计算并保存当前步骤的中间状态。
oracle_edges = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
oracle_agg = neighbor_sum(oracle_x, oracle_edges)  # 计算并保存当前步骤的中间状态。
assert torch.equal(oracle_agg, torch.tensor([[2.0], [5.0], [2.0]]))  # 用受控断言验证关键不变量。
assert not torch.equal(oracle_agg, torch.tensor([[3.0], [7.0], [6.0]]))  # 没有偷偷加中心节点
assert torch.equal(neighbor_sum(torch.ones(2, 3), torch.empty((2, 0), dtype=torch.long)), torch.zeros(2, 3))  # 用受控断言验证关键不变量。

# 非对称单向链才真正能杀死 source/target 颠倒的实现：0→1、1→2。
direction_x = torch.tensor([[1.0], [10.0], [100.0]])  # 计算并保存当前步骤的中间状态。
direction_edges = torch.tensor([[0, 1], [1, 2]], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
direction_expected = torch.tensor([[0.0], [1.0], [10.0]])  # 计算并保存当前步骤的中间状态。
assert torch.equal(neighbor_sum(direction_x, direction_edges), direction_expected)  # 用受控断言验证关键不变量。
assert not torch.equal(neighbor_sum(direction_x, direction_edges.flip(0)), direction_expected)  # 用受控断言验证关键不变量。


## 4. `GINLayer`：可学习中心权重与 MLP

GIN 的一层为

$$h_i^{(k)}=\mathrm{MLP}^{(k)}\!\left((1+\epsilon^{(k)})h_i^{(k-1)}+\sum_{j\in\mathcal N(i)}h_j^{(k-1)}\right).$$

`eps` 可以固定为 0，也可以注册为标量参数；这里选择可学习版本。MLP 不是用一个线性层敷衍，而是显式实现 `Linear → ReLU → Linear → LayerNorm`。输入 `(N,F_in)`，输出 `(N,F_out)`；单层消息聚合时间复杂度约为 $O(EF_{in})$，MLP 为 $O(NF_{in}F_h)$，稀疏边存储为 $O(E)$。

In [ ]:
class GINMLP(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_features: int, hidden_features: int, out_features: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if min(in_features, hidden_features, out_features) <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("MLP 维度必须为正")  # 遇到非法合同立即显式失败。
        self.in_features = in_features  # 计算并保存当前步骤的中间状态。
        self.fc1 = nn.Linear(in_features, hidden_features)  # 计算并保存当前步骤的中间状态。
        self.fc2 = nn.Linear(hidden_features, out_features)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(out_features)  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if x.ndim != 2 or x.shape[1] != self.in_features or not torch.isfinite(x).all():  # 按当前条件选择后续控制路径。
            raise ValueError("MLP 输入 shape 或数值非法")  # 遇到非法合同立即显式失败。
        return self.norm(self.fc2(F.relu(self.fc1(x))))  # 返回当前分支计算出的结果。

class GINLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_features: int, hidden_features: int, out_features: int,  # 定义本节可复用的核心函数。
                 initial_eps: float = 0.0, train_eps: bool = True):  # 计算并保存当前步骤的中间状态。
        super().__init__()  # 执行当前语句以推进本节示例。
        eps = torch.tensor(float(initial_eps), dtype=DTYPE)  # 计算并保存当前步骤的中间状态。
        if train_eps:  # 按当前条件选择后续控制路径。
            self.eps = nn.Parameter(eps)  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            self.register_buffer("eps", eps)  # 执行当前语句以推进本节示例。
        self.mlp = GINMLP(in_features, hidden_features, out_features)  # 计算并保存当前步骤的中间状态。
        self.in_features = in_features  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if x.ndim != 2 or x.shape[1] != self.in_features:  # 按当前条件选择后续控制路径。
            raise ValueError("GINLayer 输入特征维度不匹配")  # 遇到非法合同立即显式失败。
        aggregated = neighbor_sum(x, edge_index)  # 计算并保存当前步骤的中间状态。
        return self.mlp((1.0 + self.eps) * x + aggregated)  # 返回当前分支计算出的结果。

gin_probe = GINLayer(1, 4, 3, initial_eps=0.25)  # 计算并保存当前步骤的中间状态。
gin_out = gin_probe(oracle_x, oracle_edges)  # 计算并保存当前步骤的中间状态。
assert gin_out.shape == (3, 3) and torch.isfinite(gin_out).all()  # 用受控断言验证关键不变量。
assert isinstance(gin_probe.eps, nn.Parameter) and gin_probe.eps.requires_grad  # 用受控断言验证关键不变量。
assert math.isclose(float(gin_probe.eps.detach()), 0.25, abs_tol=1e-7)  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in gin_probe.parameters()) == 1 + (1*4+4) + (4*3+3) + 2*3  # 用受控断言验证关键不变量。

## 5. 图分类器：逐层表示与图级读出

三层 GIN 产生节点表示 $H^{(1)},H^{(2)},H^{(3)}$。每层都做 sum pooling，再把三个图向量拼接后分类，相当于保留不同感受野的信号。`forward` 接收拼接节点、块对角边和 batch 所属关系，输出 `(B,C)` logits；它不会在内部根据标签改变图结构。

GIN 对**节点编号置换**应保持图级输出不变，但这要求特征和边同时重编号。只打乱特征、不改边不是置换同一张图，而是在创建另一张有属性图。

In [ ]:
class GINClassifier(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_features: int, hidden_features: int, num_classes: int, num_layers: int = 3):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if num_layers < 1 or num_classes < 2:  # 按当前条件选择后续控制路径。
            raise ValueError("层数或类别数非法")  # 遇到非法合同立即显式失败。
        layers = []  # 计算并保存当前步骤的中间状态。
        for layer_id in range(num_layers):  # 遍历输入元素以累积或检查结果。
            in_dim = in_features if layer_id == 0 else hidden_features  # 计算并保存当前步骤的中间状态。
            layers.append(GINLayer(in_dim, hidden_features, hidden_features))  # 执行当前语句以推进本节示例。
        self.layers = nn.ModuleList(layers)  # 计算并保存当前步骤的中间状态。
        self.classifier = nn.Linear(hidden_features * num_layers, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor,  # 定义本节可复用的核心函数。
                batch: torch.Tensor, num_graphs: int) -> torch.Tensor:  # 执行当前语句以推进本节示例。
        validate_batched_graph_tensors(x, edge_index, batch, num_graphs)  # 执行当前语句以推进本节示例。
        pooled_layers = []  # 计算并保存当前步骤的中间状态。
        h = x  # 计算并保存当前步骤的中间状态。
        for layer in self.layers:  # 遍历输入元素以累积或检查结果。
            h = F.relu(layer(h, edge_index))  # 计算并保存当前步骤的中间状态。
            pooled_layers.append(sum_pool(h, batch, num_graphs))  # 执行当前语句以推进本节示例。
        return self.classifier(torch.cat(pooled_layers, dim=-1))  # 返回当前分支计算出的结果。

torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
invariant_model = GINClassifier(2, 12, 2).eval()  # 计算并保存当前步骤的中间状态。
sample = graphs[2]  # 计算并保存当前步骤的中间状态。
base = batch_graphs([sample])  # 计算并保存当前步骤的中间状态。
base_logits = invariant_model(base.x, base.edge_index, base.batch, 1)  # 计算并保存当前步骤的中间状态。

perm = torch.tensor([2, 0, 5, 1, 4, 3, 6, 7][:sample.x.shape[0]])  # 计算并保存当前步骤的中间状态。
old_to_new = torch.empty(sample.x.shape[0], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
old_to_new[perm] = torch.arange(sample.x.shape[0])  # 计算并保存当前步骤的中间状态。
perm_pairs = tuple(sorted(tuple(sorted((int(old_to_new[u]), int(old_to_new[v])))) for u, v in sample.edge_pairs))  # 计算并保存当前步骤的中间状态。
perm_sample = GraphSample("permuted", sample.x[perm], perm_pairs, sample.label, sample.split)  # 计算并保存当前步骤的中间状态。
perm_batch = batch_graphs([perm_sample])  # 计算并保存当前步骤的中间状态。
perm_logits = invariant_model(perm_batch.x, perm_batch.edge_index, perm_batch.batch, 1)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(base_logits, perm_logits, atol=2e-6)  # 用受控断言验证关键不变量。

pair = [graphs[1], graphs[20]]  # 计算并保存当前步骤的中间状态。
together = batch_graphs(pair)  # 计算并保存当前步骤的中间状态。
logits_together = invariant_model(together.x, together.edge_index, together.batch, 2)  # 计算并保存当前步骤的中间状态。
logits_separate = torch.cat([  # 计算并保存当前步骤的中间状态。
    invariant_model((one := batch_graphs([g])).x, one.edge_index, one.batch, 1) for g in pair  # 计算并保存当前步骤的中间状态。
], dim=0)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(logits_together, logits_separate, atol=2e-6)  # 用受控断言验证关键不变量。
assert logits_together.shape == (2, 2)  # 用受控断言验证关键不变量。

# 模型边界也必须 fail closed，不能只信任 batch_graphs 的调用路径。
cross_source = int((together.batch == 0).nonzero(as_tuple=False)[0])  # 计算并保存当前步骤的中间状态。
cross_target = int((together.batch == 1).nonzero(as_tuple=False)[0])  # 计算并保存当前步骤的中间状态。
forged_edges = torch.cat([together.edge_index,  # 计算并保存当前步骤的中间状态。
                          torch.tensor([[cross_source], [cross_target]], dtype=torch.long)], dim=1)  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    invariant_model(together.x, forged_edges, together.batch, 2)  # 执行当前语句以推进本节示例。
    raise AssertionError("跨图边进入模型后未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "跨图边" in str(exc)  # 用受控断言验证关键不变量。


## 6. 受控训练、checkpoint 与梯度

全部训练图组成一个稀疏 batch。每轮仅对 train 计算交叉熵；每 5 轮在 validation 上选择 checkpoint。为了让 Notebook 快速、确定地执行，模型很小且不使用 DataLoader 多进程。我们同时要求首层 `eps`、MLP 权重和分类头都收到有限非零梯度，避免“模型表面训练、消息层实际断路”。

In [ ]:
train_batch = batch_graphs(split_graphs["train"])  # 计算并保存当前步骤的中间状态。
val_batch = batch_graphs(split_graphs["val"])  # 计算并保存当前步骤的中间状态。
test_batch = batch_graphs(split_graphs["test"])  # 计算并保存当前步骤的中间状态。

torch.manual_seed(SEED + 2)  # 执行当前语句以推进本节示例。
model = GINClassifier(2, 20, 2, num_layers=3).to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.Adam(model.parameters(), lr=0.015, weight_decay=1e-5)  # 计算并保存当前步骤的中间状态。

model.train()  # 执行当前语句以推进本节示例。
initial_logits = model(train_batch.x, train_batch.edge_index, train_batch.batch, train_batch.num_graphs)  # 计算并保存当前步骤的中间状态。
initial_loss = float(F.cross_entropy(initial_logits, train_batch.labels))  # 计算并保存当前步骤的中间状态。
best_val, best_state = -1.0, None  # 计算并保存当前步骤的中间状态。
train_start = time.perf_counter()  # 计算并保存当前步骤的中间状态。
for epoch in range(141):  # 遍历输入元素以累积或检查结果。
    model.train(); optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits = model(train_batch.x, train_batch.edge_index, train_batch.batch, train_batch.num_graphs)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits, train_batch.labels)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    if epoch == 0:  # 按当前条件选择后续控制路径。
        checked = [model.layers[0].eps.grad, model.layers[0].mlp.fc1.weight.grad,  # 计算并保存当前步骤的中间状态。
                   model.classifier.weight.grad]  # 执行当前语句以推进本节示例。
        assert all(g is not None and torch.isfinite(g).all() and float(g.abs().sum()) > 0 for g in checked)  # 用受控断言验证关键不变量。
    optimizer.step()  # 执行当前语句以推进本节示例。
    if epoch % 5 == 0:  # 按当前条件选择后续控制路径。
        model.eval()  # 执行当前语句以推进本节示例。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            val_logits = model(val_batch.x, val_batch.edge_index, val_batch.batch, val_batch.num_graphs)  # 计算并保存当前步骤的中间状态。
            val_acc = float((val_logits.argmax(1) == val_batch.labels).float().mean())  # 计算并保存当前步骤的中间状态。
        if val_acc > best_val:  # 按当前条件选择后续控制路径。
            best_val = val_acc  # 计算并保存当前步骤的中间状态。
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # 计算并保存当前步骤的中间状态。

assert best_state is not None  # 用受控断言验证关键不变量。
model.load_state_dict(best_state); model.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_train_loss = float(F.cross_entropy(  # 计算并保存当前步骤的中间状态。
        model(train_batch.x, train_batch.edge_index, train_batch.batch, train_batch.num_graphs),  # 执行当前语句以推进本节示例。
        train_batch.labels))  # 执行当前语句以推进本节示例。
    test_logits = model(test_batch.x, test_batch.edge_index, test_batch.batch, test_batch.num_graphs)  # 计算并保存当前步骤的中间状态。
    test_acc = float((test_logits.argmax(1) == test_batch.labels).float().mean())  # 计算并保存当前步骤的中间状态。
train_seconds = time.perf_counter() - train_start  # 计算并保存当前步骤的中间状态。
assert final_train_loss < initial_loss * 0.35  # 用受控断言验证关键不变量。
assert best_val >= 0.99 and test_acc >= 0.99  # 用受控断言验证关键不变量。
assert train_seconds < 12.0  # 用受控断言验证关键不变量。
print({"initial_loss": round(initial_loss, 4), "final_loss": round(final_train_loss, 4),  # 执行当前语句以推进本节示例。
       "val_acc": best_val, "test_acc": test_acc, "seconds": round(train_seconds, 3)})  # 执行当前语句以推进本节示例。

## 7. 常见失败模式与生产边界

- **把 sum 改成 mean**：会削弱对多重集合计数的区分；是否使用 mean 应由任务假设决定，不能悄悄替换。
- **重复加入两个方向**：若原始无向边已展开，再次展开会把消息加倍。批处理函数只接受去重的无向 pair。
- **BatchNorm 污染**：图分开/合并执行时，BatchNorm 的统计会改变。本实现用逐节点 LayerNorm，使批隔离 oracle 更直接。
- **过平滑与过拟合**：层数增加并不总是更好；生产上需按图规模监控表征方差、图大小分桶指标和 OOD 拒识。
- **超大图**：Python 边循环只适合教学。生产应换成经过验证的稀疏 kernel，但必须保留方向、去重和跨图隔离测试。

权限过滤应在建图之前完成。图快照、特征模式、代码版本和权重必须共同绑定；仅保存一个 `.pt` 文件不足以复现实验。

In [ ]:
def state_hash(module: nn.Module) -> str:  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for name, tensor in sorted(module.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        digest.update(name.encode("utf-8")); digest.update(tensor.detach().cpu().contiguous().numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()[:20]  # 返回当前分支计算出的结果。

graph_snapshot = [{"id": g.graph_id, "n": int(g.x.shape[0]), "edges": list(g.edge_pairs),  # 计算并保存当前步骤的中间状态。
                   "label": g.label, "split": g.split} for g in graphs]  # 执行当前语句以推进本节示例。
artifact = {  # 计算并保存当前步骤的中间状态。
    "architecture": "GINClassifier-3layer-sum-readout-v1",  # 执行当前语句以推进本节示例。
    "seed": SEED,  # 执行当前语句以推进本节示例。
    "feature_schema_hash": canonical_hash({"columns": ["constant", "local_parity"], "dtype": "float32"}),  # 执行当前语句以推进本节示例。
    "graph_snapshot_hash": canonical_hash(graph_snapshot),  # 执行当前语句以推进本节示例。
    "split_hash": canonical_hash({s: [g.graph_id for g in split_graphs[s]] for s in split_graphs}),  # 执行当前语句以推进本节示例。
    "state_hash": state_hash(model),  # 执行当前语句以推进本节示例。
    "metrics": {"best_val_accuracy": best_val, "test_accuracy": test_acc},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact["artifact_id"] = canonical_hash(artifact)  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class InferenceContext:  # 定义承载本节状态与行为的数据结构。
    scopes: frozenset[str]  # 执行当前语句以推进本节示例。
    expected_artifact_id: str  # 执行当前语句以推进本节示例。

def predict_graph(graph: GraphSample, ctx: InferenceContext) -> int:  # 定义本节可复用的核心函数。
    if "graph:predict" not in ctx.scopes:  # 按当前条件选择后续控制路径。
        raise PermissionError("缺少 graph:predict")  # 遇到非法合同立即显式失败。
    if ctx.expected_artifact_id != artifact["artifact_id"]:  # 按当前条件选择后续控制路径。
        raise RuntimeError("请求绑定的 artifact 与当前服务不一致")  # 遇到非法合同立即显式失败。
    item = batch_graphs([graph])  # 计算并保存当前步骤的中间状态。
    with torch.no_grad():  # 在受管理的上下文中执行操作。
        return int(model(item.x, item.edge_index, item.batch, 1).argmax(1).item())  # 返回当前分支计算出的结果。

ok_ctx = InferenceContext(frozenset({"graph:predict"}), artifact["artifact_id"])  # 计算并保存当前步骤的中间状态。
assert predict_graph(graphs[-1], ok_ctx) in (0, 1)  # 用受控断言验证关键不变量。
assert len({artifact["feature_schema_hash"], artifact["graph_snapshot_hash"],  # 用受控断言验证关键不变量。
            artifact["split_hash"], artifact["state_hash"]}) == 4  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    predict_graph(graphs[-1], InferenceContext(frozenset(), artifact["artifact_id"]))  # 执行当前语句以推进本节示例。
    raise AssertionError("缺少权限仍可推理")  # 遇到非法合同立即显式失败。
except PermissionError as exc:  # 捕获预期异常并验证失败分支。
    assert "graph:predict" in str(exc)  # 用受控断言验证关键不变量。
print({"artifact_id": artifact["artifact_id"], "state_hash": artifact["state_hash"]})  # 执行当前语句以推进本节示例。

## 8. 面试复盘与论文来源

一个完整回答应覆盖：为什么 GIN 使用 sum、`ε` 的作用、消息方向、多图块对角 batch、图级读出、置换不变性、切分协议，以及如何把权重绑定到图快照与特征模式。只写 `class GIN(nn.Module)` 而没有 oracle，无法证明实现的其实是 GIN。

主要来源：Xu, Hu, Leskovec & Jegelka, [**How Powerful are Graph Neural Networks?**](https://arxiv.org/abs/1810.00826), ICLR 2019；Morris et al., [**Weisfeiler and Leman Go Neural**](https://arxiv.org/abs/1810.02244), AAAI 2019。前者将 GIN 与 Weisfeiler–Lehman 图同构测试的表达力联系起来；本 Notebook 是小规模工程复现，不声称覆盖论文的全部理论与实验。